# MT Model Training with LoRA + BF16 - Sequential Fine-Tuning Experiments (NLLB-200)

This notebook tests whether **sequential fine-tuning with LoRA adapters** (similar language → baseline) improves MT performance compared to **direct fine-tuning with LoRA** (baseline only) using the **NLLB-200 distilled 600M model**.

## Why NLLB-200?
- **200 Languages:** Supports 200 languages including many low-resource Philippine languages
- **Distilled 600M:** Smaller, faster model (600M parameters) vs original 1.3B/3.3B
- **Better Philippine Language Support:** NLLB has better tokenization for Philippine languages
- **Native Language Codes:** Uses standard language codes (e.g., `tgl_Latn` for Tagalog)

## Experimental Design: Sequential Fine-Tuning with LoRA

For each target language, we create TWO models:

### 1. Baseline Models (Direct LoRA Training)
- **Start:** NLLB-200-distilled-600M (pretrained)
- **Train:** Apply LoRA adapters and train Distant language → Target language (e.g., `en→war`)
- **Result:** Baseline Model with LoRA adapters

### 2. Experimental Models (Sequential LoRA Fine-Tuning)
- **Start:** NLLB-200-distilled-600M (pretrained)
- **Step 1:** Apply LoRA adapters and train Distant language → Similar language (e.g., `en→ceb`)
- **Step 2:** Load Stage 1 LoRA adapters and **continue training** on Distant language → Target language (e.g., `en→war`)
- **Result:** Experimental Model (with similarity transfer via LoRA)

### Target Language
**Waray (war)**
   - Baseline: `en→war` only
   - Experimental: `en→ceb` THEN `en→war`
**Cebuano (ceb)**
   - Baseline: `en→ceb` only
   - Experimental: `en→war` THEN `en→ceb`


## Install Required Packages

Run this cell first if packages are not installed. Note the addition of `peft` for LoRA support.

In [1]:
# Uncomment and run if needed
# !pip install transformers datasets evaluate sacrebleu torch sentencepiece accelerate peft

## Imports

In [2]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import Dataset, DatasetDict
import evaluate
import numpy as np
import torch
from pathlib import Path
import json


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please upd

## Configuration

Set up training configurations for all language pairs, including LoRA-specific parameters and BF16 optimization with NLLB-200 language codes.

In [3]:
# Model configuration
MODEL_NAME = "facebook/nllb-200-distilled-600M"
MAX_LENGTH = 128
BATCH_SIZE = 4  
LEARNING_RATE = 1e-5  
BASE_LINE_EPOCHS = 6  # For baseline training (Stage 2)
NUM_EPOCHS_STAGE1 = 6  # For similar language training (Stage 1)
NUM_EPOCHS_STAGE2 = 6  # For baseline training (Stage 2)

# BF16 Optimization
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# LoRA configuration
LORA_R = 16 
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]  # attention layers

# Directories
DATA_DIR = Path("../data/splits")
OUTPUT_DIR = Path("../models")
LOGS_DIR = Path("../logs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# NLLB-200 language code mapping
NLLB_LANG_CODES = {
    "en": "eng_Latn",
    "tl": "tgl_Latn",  
    "war": "war_Latn",   
    "ceb": "ceb_Latn",  
}

# Experimental configurations with NLLB language codes
EXPERIMENTS = {
    "waray": {
        "target": "war",
        "baseline_pair": "en-war",
        "similar_pair": "en-ceb",
        "baseline_config": {
            "src_lang": NLLB_LANG_CODES["en"],
            "tgt_lang": NLLB_LANG_CODES["war"],
        },
        "similar_config": {
            "src_lang": NLLB_LANG_CODES["en"],
            "tgt_lang": NLLB_LANG_CODES["ceb"],
        }
    },
    "cebuano":{
        "target": "cebuano",
        "baseline_pair": "en-ceb",
        "similar_pair": "en-war",
        "baseline_config": {
            "src_lang": NLLB_LANG_CODES["en"],
            "tgt_lang": NLLB_LANG_CODES["ceb"],
        },
        "similar_config": {
            "src_lang": NLLB_LANG_CODES["en"],
            "tgt_lang": NLLB_LANG_CODES["war"],
        }
    }
}

print("Experimental Configuration (NLLB-200 + LoRA + BF16):")
print(f"  Model: {MODEL_NAME}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  BF16 enabled: {USE_BF16}")
print(f"  LoRA rank (r): {LORA_R}")
print(f"  LoRA alpha: {LORA_ALPHA}")
print(f"  LoRA target modules: {LORA_TARGET_MODULES}")
print(f"  Epochs (Stage 1): {NUM_EPOCHS_STAGE1}")
print(f"  Epochs (Stage 2): {NUM_EPOCHS_STAGE2}")
print(f"\nNLLB Language Codes:")
for code, nllb_code in NLLB_LANG_CODES.items():
    print(f"  {code} → {nllb_code}")
print(f"\nTarget Languages: {len(EXPERIMENTS)}")
for lang, config in EXPERIMENTS.items():
    print(f"  - {lang.capitalize()}: {config['baseline_pair']} (baseline) vs {config['similar_pair']}→{config['baseline_pair']} (sequential)")

Experimental Configuration (NLLB-200 + LoRA + BF16):
  Model: facebook/nllb-200-distilled-600M
  Batch size: 4
  Learning rate: 1e-05
  BF16 enabled: True
  LoRA rank (r): 16
  LoRA alpha: 32
  LoRA target modules: ['q_proj', 'v_proj', 'k_proj', 'o_proj']
  Epochs (Stage 1): 6
  Epochs (Stage 2): 6

NLLB Language Codes:
  en → eng_Latn
  tl → tgl_Latn
  war → war_Latn
  ceb → ceb_Latn

Target Languages: 2
  - Waray: en-war (baseline) vs en-ceb→en-war (sequential)
  - Cebuano: en-ceb (baseline) vs en-war→en-ceb (sequential)


## Helper Functions

Functions to load data, preprocess, evaluate models, and apply LoRA adapters for NLLB-200.

In [4]:
def load_data_for_pair(pair_name):
    """Load train and dev splits for a language pair."""
    src_code, tgt_code = pair_name.split("-")
    pair_dir = DATA_DIR / pair_name
    
    with open(pair_dir / f"train.{src_code}", "r", encoding="utf-8") as f:
        train_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"train.{tgt_code}", "r", encoding="utf-8") as f:
        train_tgt = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"dev.{src_code}", "r", encoding="utf-8") as f:
        dev_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"dev.{tgt_code}", "r", encoding="utf-8") as f:
        dev_tgt = [line.strip() for line in f.readlines()]
    
    train_dataset = Dataset.from_dict({"src": train_src, "tgt": train_tgt})
    dev_dataset = Dataset.from_dict({"src": dev_src, "tgt": dev_tgt})
    
    dataset_dict = DatasetDict({"train": train_dataset, "validation": dev_dataset})
    
    print(f"Loaded {pair_name}: Train={len(train_dataset)}, Dev={len(dev_dataset)}")
    return dataset_dict


def create_preprocess_function(tokenizer, src_lang, tgt_lang, max_length):
    """Create preprocessing function for tokenization with NLLB-200."""
    def preprocess(batch):
        # NLLB uses src_lang and tgt_lang attributes
        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            batch["src"],   
            truncation=True,
            padding="max_length",
            max_length=max_length
        )
        
        # Set target language and tokenize
        tokenizer.tgt_lang = tgt_lang
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                batch["tgt"],
                truncation=True,
                padding="max_length",
                max_length=max_length
            )
        inputs["labels"] = labels["input_ids"]
        return inputs
    return preprocess


def create_compute_metrics(tokenizer): 
    """Create function to compute BLEU score during evaluation.""" 
    bleu = evaluate.load("sacrebleu") 
    def compute_metrics(eval_pred): 
        preds, labels = eval_pred 
        if isinstance(preds, tuple): 
            preds = preds[0] 
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id) 
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True, clean_up_tokenization_spaces=True) 
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True, clean_up_tokenization_spaces=True) 
        result = bleu.compute( 
            predictions=decoded_preds, 
            references=[[label] for label in decoded_labels] 
        ) 
        return {"bleu": result["score"]} 
    return compute_metrics


def apply_lora_to_model(model):
    """Apply LoRA adapters to the NLLB model."""
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()
    return peft_model


print("Helper functions defined")

Helper functions defined


## Test Preprocessing and Tokenization

Run a quick test to see how the data is tokenized before training.

In [5]:
# Test preprocessing with one language pair
test_pair = "en-ceb" 
test_config = EXPERIMENTS["cebuano"]["baseline_config"]

print(f"Testing preprocessing for: {test_pair}")
print(f"  Source language: {test_config['src_lang']}")
print(f"  Target language: {test_config['tgt_lang']}")

# Load tokenizer
test_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load a small sample of data
print("\n1. Loading data...")
dataset = load_data_for_pair(test_pair)

print("\n2. Sample raw data (first 3 examples):")
for i in range(min(3, len(dataset['train']))):
    print(f"\n  Example {i+1}:")
    print(f"    Source: {dataset['train'][i]['src'][:100]}...")
    print(f"    Target: {dataset['train'][i]['tgt'][:100]}...")

# Create preprocessing function
print("\n3. Creating preprocessing function...")
preprocess_fn = create_preprocess_function(
    test_tokenizer, 
    test_config['src_lang'], 
    test_config['tgt_lang'], 
    MAX_LENGTH
)

# Preprocess a small batch
print("\n4. Preprocessing samples...")
sample_batch = {
    'src': dataset['train'][:3]['src'],
    'tgt': dataset['train'][:3]['tgt']
}
tokenized_batch = preprocess_fn(sample_batch)

print("\n5. Tokenized data:")
for i in range(len(tokenized_batch['input_ids'])):
    print(f"\n  Example {i+1}:")
    print(f"    Input IDs shape: {len(tokenized_batch['input_ids'][i])}")
    print(f"    Input IDs (first 20): {tokenized_batch['input_ids'][i][:20]}")
    print(f"    Labels shape: {len(tokenized_batch['labels'][i])}")
    print(f"    Labels (first 20): {tokenized_batch['labels'][i][:20]}")
    
     # Show actual tokenized text (tokens)
    input_tokens = test_tokenizer.convert_ids_to_tokens(tokenized_batch['input_ids'][i])
    label_tokens = test_tokenizer.convert_ids_to_tokens(
        [label_id if label_id != -100 else test_tokenizer.pad_token_id for label_id in tokenized_batch['labels'][i]]
    )
    print(f"    Input tokens: {input_tokens[:20]}")
    print(f"    Label tokens: {label_tokens[:20]}")
    
    # Decode to verify
    decoded_input = test_tokenizer.decode(tokenized_batch['input_ids'][i], skip_special_tokens=False)
    decoded_label = test_tokenizer.decode(
        [label_id if label_id != -100 else test_tokenizer.pad_token_id 
         for label_id in tokenized_batch['labels'][i]], 
        skip_special_tokens=False
    )
    print(f"    Decoded input: {decoded_input[:100]}...")
    print(f"    Decoded label: {decoded_label[:100]}...")

print("\n6. Dataset statistics:")
print(f"  Train size: {len(dataset['train'])}")
print(f"  Validation size: {len(dataset['validation'])}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Vocab size: {len(test_tokenizer)}")

print("\nPreprocessing test complete")

Testing preprocessing for: en-ceb
  Source language: eng_Latn
  Target language: ceb_Latn

1. Loading data...
Loaded en-ceb: Train=3634, Dev=454

2. Sample raw data (first 3 examples):

  Example 1:
    Source: When Saul sent the men to capture David, Michal said, “He is ill.”...
    Target: Ug sa diha nga si Saul nagpadala ug mga sinugo aron sa pagkuha kang David, si Michal miingon, “Siya ...

  Example 2:
    Source: Otherwise, when you eat and are satisfied, when you build fine houses and settle down,...
    Target: Tingali unya ug sa dihang makakaon na kamo ug mabusog ug makatukod na ug matahom nga mga balay ug ma...

  Example 3:
    Source: Israel sent messengers to say to Sihon king of the Amorites:...
    Target: Unya ang Israel nagpadala ug mga sulugoon ngadto kang Sihon nga hari sa mga Amorihanon ug miingon,...

3. Creating preprocessing function...

4. Preprocessing samples...

5. Tokenized data:

  Example 1:
    Input IDs shape: 128
    Input IDs (first 20): [256047, 22502

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


## Training Functions

Functions to train single stages and run complete experiments with LoRA and BF16 for NLLB-200.

In [6]:
def train_single_stage(pair_name, config, model_name_or_path, output_subdir, num_epochs, stage_name="", is_stage2=False):
    """Train a single stage with LoRA adapters and BF16 using NLLB-200."""
    print("\n" + "="*80)
    print(f"Training: {pair_name.upper()}")
    if stage_name:
        print(f"Stage: {stage_name}")
    print("="*80)
    
    # Load tokenizer
    print("\n1. Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.src_lang = config['src_lang']
    tokenizer.tgt_lang = config['tgt_lang']
    
    # Set forced_bos_token_id for target language generation
    forced_bos_token_id = None
    if hasattr(tokenizer, 'lang_code_to_id'):
        forced_bos_token_id = tokenizer.lang_code_to_id.get(config['tgt_lang'])
        print(f"   Target language token ID (forced_bos_token_id): {forced_bos_token_id}")
    else:
        # Fallback: try to get token ID from convert_tokens_to_ids
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(config['tgt_lang'])
        if forced_bos_token_id == tokenizer.unk_token_id:
            forced_bos_token_id = None
        print(f"   Target language token ID (fallback forced_bos_token_id): {forced_bos_token_id}")
    
    # Load model
    print("\n2. Loading model...")
    if is_stage2:
        # Stage 2: Load base model then load Stage 1 adapters
        print(f"   Loading Stage 1 adapters from: {model_name_or_path}")
        base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        model = PeftModel.from_pretrained(base_model, model_name_or_path, is_trainable=True)
        
        # Explicitly enable training mode and mark adapters as trainable
        model.train()
        model.base_model.train()
        
        # Print trainable parameters to verify
        model.print_trainable_parameters()
        
        print("   Stage 1 adapters loaded successfully")
        print("   Model configured for continued training")
    else:
        # Stage 1 or Baseline: Load base model and apply new LoRA
        print(f"   Loading base model: {MODEL_NAME}")
        base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        model = apply_lora_to_model(base_model)
    
    # Load dataset
    print("\n3. Loading dataset...")
    dataset = load_data_for_pair(pair_name)
    
    # Tokenize
    print("\n4. Tokenizing dataset...")
    print(f"   Source language: {config['src_lang']}")
    print(f"   Target language: {config['tgt_lang']}")
    preprocess_fn = create_preprocess_function(
        tokenizer, config['src_lang'], config['tgt_lang'], MAX_LENGTH
    )
    tokenized_dataset = dataset.map(preprocess_fn, batched=True)
    
    # Training arguments with BF16
    output_dir = OUTPUT_DIR / output_subdir
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="linear",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=num_epochs,
        save_strategy="epoch",
        save_total_limit=2,
        predict_with_generate=True,
        logging_dir=str(LOGS_DIR / output_subdir),
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        bf16=USE_BF16,  
        fp16=False,  
        report_to="none",
        warmup_ratio=0.1,
        generation_max_length=MAX_LENGTH,
    )
    
    # Set forced_bos_token_id in model config for generation
    if forced_bos_token_id is not None:
        model.config.forced_bos_token_id = forced_bos_token_id
        print(f"\n    Set model.config.forced_bos_token_id = {forced_bos_token_id} ({config['tgt_lang']})")
    else:
        print(f"\n    Warning: Could not set forced_bos_token_id for {config['tgt_lang']}")
    
    # Create trainer
    print("\n5. Setting up trainer...")
    print(f"   Using BF16: {USE_BF16}")
    compute_metrics_fn = create_compute_metrics(tokenizer)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_fn,
    )
    
    # Train
    print("\n6. Starting training...")
    train_result = trainer.train()
    
    # Save model (LoRA adapters)
    print("\n7. Saving LoRA adapters and tokenizer...")
    final_model_path = output_dir / "final_model"
    model.save_pretrained(str(final_model_path))
    tokenizer.save_pretrained(str(final_model_path))
    
    # Evaluate
    print("\n8. Final evaluation...")
    eval_results = trainer.evaluate()
    
    # Save results
    results = {
        "pair": pair_name,
        "stage": stage_name,
        "model_name": MODEL_NAME,
        "model_source": model_name_or_path,
        "bf16_enabled": USE_BF16,
        "src_lang": config['src_lang'],
        "tgt_lang": config['tgt_lang'],
        "lora_config": {
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_modules": LORA_TARGET_MODULES
        },
        "train_results": {
            "train_loss": train_result.training_loss,
            "train_runtime": train_result.metrics["train_runtime"],
        },
        "eval_results": eval_results
    }
    
    results_file = output_dir / "training_results.json"
    with open(results_file, "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"\n Training complete")
    print(f"  Final BLEU: {eval_results['eval_bleu']:.2f}")
    print(f"  Training time: {train_result.metrics['train_runtime']:.1f}s")
    print(f"  LoRA adapters saved to: {final_model_path}")
    
    return results, str(final_model_path)


def train_experiment(target_lang_name, experiment_config):
    """Run complete experiment for one target language with LoRA and BF16 using NLLB-200."""
    print("\n" + "#"*80)
    print(f"# EXPERIMENT (NLLB-200 + LoRA + BF16): {target_lang_name.upper()}")
    print("#"*80)
    
    baseline_pair = experiment_config['baseline_pair']
    similar_pair = experiment_config['similar_pair']
    all_results = {}
    
    # BASELINE: Direct LoRA training on distant → target
    print(f"\n{'='*80}")
    print(f"BASELINE (NLLB-200 + LoRA + BF16): {baseline_pair}")
    print("="*80)
    baseline_results, baseline_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_baseline_nllb_lora_bf16",
        num_epochs=BASE_LINE_EPOCHS,
        stage_name="Baseline",
        is_stage2=False
    )
    all_results['baseline'] = baseline_results
    
    # EXPERIMENTAL - STAGE 1: LoRA on similar → target
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL - STAGE 1: {similar_pair}")
    print("="*80)
    stage1_results, stage1_model_path = train_single_stage(
        pair_name=similar_pair,
        config=experiment_config['similar_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_experimental_stage1_nllb_lora_bf16",
        num_epochs=NUM_EPOCHS_STAGE1,
        stage_name="Stage 1 (Similar Language)",
        is_stage2=False
    )
    all_results['experimental_stage1'] = stage1_results
    
    # EXPERIMENTAL - STAGE 2: Continue LoRA from Stage 1 on distant → target
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL - STAGE 2: {baseline_pair}")
    print(f"Continuing from Stage 1 LoRA adapters")
    print("="*80)
    stage2_results, stage2_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=stage1_model_path,
        output_subdir=f"{target_lang_name}_experimental_stage2_nllb_lora_bf16",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Stage 2 (Baseline after Similar)",
        is_stage2=True
    )
    all_results['experimental_stage2'] = stage2_results
    
    # SUMMARY
    print("\n" + "="*80)
    print(f"EXPERIMENT COMPLETE (NLLB-200 + LoRA + BF16): {target_lang_name.upper()}")
    print("="*80)
    print(f"\nBaseline (NLLB LoRA + BF16): {all_results['baseline']['eval_results']['eval_bleu']:.2f} BLEU")
    print(f"Experimental Stage 1: {all_results['experimental_stage1']['eval_results']['eval_bleu']:.2f} BLEU")
    print(f"Experimental Stage 2: {all_results['experimental_stage2']['eval_results']['eval_bleu']:.2f} BLEU")
    
    improvement = all_results['experimental_stage2']['eval_results']['eval_bleu'] - all_results['baseline']['eval_results']['eval_bleu']
    print(f"\nImprovement: {improvement:+.2f} BLEU points")
    if improvement > 0:
        print("Sequential LoRA fine-tuning with NLLB IMPROVED performance")
    elif improvement < 0:
        print("Sequential LoRA fine-tuning with NLLB DEGRADED performance")
    else:
        print("= No difference in performance")
    
    # Save summary
    summary_file = OUTPUT_DIR / f"{target_lang_name}_nllb_lora_bf16_experiment_summary.json"
    with open(summary_file, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSummary saved to: {summary_file}")
    
    return all_results


print("Training functions defined")

Training functions defined


## Run Single Experiment (NLLB-200 + LoRA + BF16)

Test with one target language first to verify the NLLB-200 + LoRA + BF16 setup.

In [7]:
# Run one experiment (uncomment to test)
target_lang = "cebuano"
results = train_experiment(target_lang, EXPERIMENTS[target_lang])


################################################################################
# EXPERIMENT (NLLB-200 + LoRA + BF16): CEBUANO
################################################################################

BASELINE (NLLB-200 + LoRA + BF16): en-ceb

Training: EN-CEB
Stage: Baseline

1. Loading tokenizer...
   Target language token ID (fallback forced_bos_token_id): 256035

2. Loading model...
   Loading base model: facebook/nllb-200-distilled-600M
trainable params: 3,538,944 || all params: 618,612,736 || trainable%: 0.5721

3. Loading dataset...
Loaded en-ceb: Train=3634, Dev=454

4. Tokenizing dataset...
   Source language: eng_Latn
   Target language: ceb_Latn


Map:   0%|          | 0/3634 [00:00<?, ? examples/s]

Map:   0%|          | 0/454 [00:00<?, ? examples/s]


    Set model.config.forced_bos_token_id = 256035 (ceb_Latn)

5. Setting up trainer...
   Using BF16: True


C:\Users\user\AppData\Local\Temp\ipykernel_21564\1808460003.py:99: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


...

C:\Users\user\AppData\Local\Temp\ipykernel_21564\1808460003.py:99: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(



6. Starting training...


Epoch,Training Loss,Validation Loss,Bleu
1,6.972600,6.602429,22.436750
2,6.759300,6.520584,27.171264
3,6.691600,6.501097,27.705508
4,6.659500,6.486491,28.251358
5,6.642900,6.483432,28.179190
6,6.634300,6.482828,28.181292



7. Saving LoRA adapters and tokenizer...

8. Final evaluation...



 Training complete
  Final BLEU: 28.25
  Training time: 2448.2s
  LoRA adapters saved to: ..\models\cebuano_experimental_stage2_nllb_lora_bf16\final_model

EXPERIMENT COMPLETE (NLLB-200 + LoRA + BF16): CEBUANO

Baseline (NLLB LoRA + BF16): 27.23 BLEU
Experimental Stage 1: 28.02 BLEU
Experimental Stage 2: 28.25 BLEU

Improvement: +1.02 BLEU points
Sequential LoRA fine-tuning with NLLB IMPROVED performance

Summary saved to: ..\models\cebuano_nllb_lora_bf16_experiment_summary.json


In [8]:
target_lang = "waray"
results = train_experiment(target_lang, EXPERIMENTS[target_lang])


################################################################################
# EXPERIMENT (NLLB-200 + LoRA + BF16): WARAY
################################################################################

BASELINE (NLLB-200 + LoRA + BF16): en-war

Training: EN-WAR
Stage: Baseline

1. Loading tokenizer...
   Target language token ID (fallback forced_bos_token_id): 256194

2. Loading model...
   Loading base model: facebook/nllb-200-distilled-600M
trainable params: 3,538,944 || all params: 618,612,736 || trainable%: 0.5721

3. Loading dataset...
Loaded en-war: Train=3634, Dev=454

4. Tokenizing dataset...
   Source language: eng_Latn
   Target language: war_Latn


Map:   0%|          | 0/3634 [00:00<?, ? examples/s]

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/454 [00:00<?, ? examples/s]


    Set model.config.forced_bos_token_id = 256194 (war_Latn)

5. Setting up trainer...
   Using BF16: True


...


6. Starting training...


Epoch,Training Loss,Validation Loss,Bleu
1,6.757800,6.482359,25.281467
2,6.651000,6.453398,27.498836
3,6.611400,6.439152,27.557668
4,6.588900,6.436836,28.525783
5,6.577400,6.435105,28.513992
6,6.571100,6.433316,28.529688


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1a502bcf-7ce6-4b41-8c52-3d5c50245906)')' thrown while requesting HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json
Retrying in 1s [Retry 1/5].



7. Saving LoRA adapters and tokenizer...

8. Final evaluation...



 Training complete
  Final BLEU: 28.53
  Training time: 7633.2s
  LoRA adapters saved to: ..\models\waray_experimental_stage2_nllb_lora_bf16\final_model

EXPERIMENT COMPLETE (NLLB-200 + LoRA + BF16): WARAY

Baseline (NLLB LoRA + BF16): 27.87 BLEU
Experimental Stage 1: 27.27 BLEU
Experimental Stage 2: 28.53 BLEU

Improvement: +0.66 BLEU points
Sequential LoRA fine-tuning with NLLB IMPROVED performance

Summary saved to: ..\models\waray_nllb_lora_bf16_experiment_summary.json
